# v7: Sequence-Level Diffusion Routing
## Denoising IS Cross-Position Mixing

**Author:** David Ledbetter  
**Date:** 03/13/2026  
**Lineage:** Builds on v1-v6, unifies cross-position mixing with Langevin settling

---

### The Core Idea

v1-v6 treated cross-position mixing and Langevin settling as **separate stages**:  
first mix information across positions (causal conv), then settle each token independently.

v7 unifies them: the Langevin denoising loop operates on the **entire sequence**,  
with a causal score function that mixes cross-position information at **every settling step**.

Diffusion models are built to denoise — to predict clean states from noisy ones  
by learning the intricate sub-interactions in the data. We define a score function  
across the sequence that does exactly this: at each Langevin step, every position  
receives context from all causal positions, and the denoising direction incorporates  
that context when settling toward sparse attractors.

### What Changes from v4-v6

| v4-v6 | v7 |
|---|---|
| Separate stages: mix then settle | **Unified: mix INSIDE each settle step** |
| Langevin operates per-token | **Langevin operates on full sequence** |
| Score = hopfield_grad(x_t) | **Score = hopfield_grad(causal_mix(x_0..t))** |
| Cross-position info applied once | **Cross-position info refined every step** |
| Routing fixed during settling | **Routing sharpens WITH settling** |

### The Annealing-Routing Connection

At high temperature (early Langevin steps): states are diffuse, mixing is broad.  
At low temperature (late Langevin steps): states are sparse, mixing is selective.  
The annealing schedule that controls sparsity now also controls routing precision.  
Denoising and routing are the same process.

In [31]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from dataclasses import dataclass
import math
from tqdm.notebook import tqdm

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cpu


---
## 1. Configuration

In [32]:
@dataclass
class ArchitectureConfig:
    """Configuration for the Sequence-Level Diffusion Routing Architecture."""

    # --- Fiber Bundle Geometry ---
    fiber_dim: int = 128
    n_subbundles: int = 8
    manifold_dim: int = 64

    # --- Vocabulary & Embedding ---
    vocab_size: int = 256
    max_seq_len: int = 128

    # --- Sparse Dictionary ---
    n_dictionary_atoms: int = 512
    k_wta: int = 64

    # --- Spectral Transport ---
    diffusion_coeff: float = 0.01
    n_blocks: int = 3

    # --- Sequence-Level Langevin (v7 core) ---
    langevin_steps: int = 7
    langevin_lr: float = 0.1
    beta_init: float = 1.0
    beta_final: float = 10.0

    # --- Sparsity ---
    sparsity_lambda: float = 0.3
    inhibition_gamma: float = 0.1

    # --- Training ---
    learning_rate: float = 5e-4
    dropout: float = 0.05
    batch_size: int = 32
    n_epochs: int = 150

    @property
    def subbundle_dim(self) -> int:
        assert self.fiber_dim % self.n_subbundles == 0
        return self.fiber_dim // self.n_subbundles


config = ArchitectureConfig()
print(f"Fiber dim: {config.fiber_dim} = {config.n_subbundles} subbundles x {config.subbundle_dim} dims each")
print(f"Langevin steps: {config.langevin_steps} (cross-position mixing at EVERY step)")
print(f"Blocks: {config.n_blocks}")

Fiber dim: 128 = 8 subbundles x 16 dims each
Langevin steps: 7 (cross-position mixing at EVERY step)
Blocks: 3


---
## 2. Foundation: Sparse Primitives and Embedding

In [33]:
def soft_threshold(x: torch.Tensor, lam: float) -> torch.Tensor:
    """Proximal operator for L1 sparsity — lateral cortical inhibition."""
    return torch.sign(x) * F.relu(torch.abs(x) - lam)


def k_wta(x: torch.Tensor, k: int) -> torch.Tensor:
    """k-Winner-Take-All: retains top-k values, zeros the rest."""
    topk_vals, topk_idx = torch.topk(x, k, dim=-1)
    out = torch.zeros_like(x)
    out.scatter_(-1, topk_idx, topk_vals)
    return out


class SparseTokenEmbedding(nn.Module):
    """Maps vocabulary tokens to sparse fiber sections."""

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.fiber_dim)
        self.topk_per_subbundle = max(1, cfg.subbundle_dim // 4)
        self.manifold_coords = nn.Embedding(cfg.max_seq_len, cfg.manifold_dim)

    def forward(self, token_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T = token_ids.shape
        x_dense = self.embedding(token_ids)

        chunks = x_dense.chunk(self.cfg.n_subbundles, dim=-1)
        sparse_chunks = []
        for chunk in chunks:
            topk_vals, topk_idx = torch.topk(
                chunk.abs(), self.topk_per_subbundle, dim=-1
            )
            mask = torch.zeros_like(chunk)
            mask.scatter_(-1, topk_idx, 1.0)
            sparse_chunks.append(chunk * mask)

        x_sparse = torch.cat(sparse_chunks, dim=-1)
        positions = torch.arange(T, device=token_ids.device)
        q_coords = self.manifold_coords(positions).unsqueeze(0).expand(B, -1, -1)

        return x_sparse, q_coords


embed = SparseTokenEmbedding(config)
test_ids = torch.randint(0, config.vocab_size, (2, 8))
x_sparse, q_coords = embed(test_ids)

sparsity = (x_sparse == 0).float().mean().item()
print(f"Sparse embedding: {x_sparse.shape}, sparsity {sparsity:.1%}")
print(f"Active dims per token: {(x_sparse != 0).float().sum(-1).mean().item():.0f} / {config.fiber_dim}")

Sparse embedding: torch.Size([2, 8, 128]), sparsity 75.0%
Active dims per token: 32 / 128


---
## 3. Gauge Connection

Position-based MLP gauge field. Handles per-fiber transport geometry  
(advection-diffusion within each token's fiber space).

In [34]:
class GaugeConnection(nn.Module):
    """MLP-based gauge field for per-fiber transport phases."""

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.connection_net = nn.Sequential(
            nn.Linear(cfg.manifold_dim * 2, cfg.fiber_dim),
            nn.Tanh(),
        )
        self.log_diffusion = nn.Parameter(
            torch.full((cfg.fiber_dim,), math.log(cfg.diffusion_coeff))
        )

    def forward(self, q_source, q_target):
        q_pair = torch.cat([q_source, q_target], dim=-1)
        phase = self.connection_net(q_pair) * math.pi

        d = self.cfg.fiber_dim
        omega = torch.fft.fftfreq(d, device=q_source.device) * 2 * math.pi
        D = self.log_diffusion.exp()
        diffusion_kernel = torch.exp(-D * omega ** 2)
        diffusion_kernel = diffusion_kernel.expand_as(phase)

        return phase, diffusion_kernel

---
## 4. Dynamic Memory Bank

In [35]:
class DynamicMemoryBank(nn.Module):
    """Geometrically-gated overcomplete sparse dictionary.

    === v7.2 FIX: Content-aware routing ===

    v7.0-v7.1 routed on position q alone. But the synthetic task has
    content-dependent patterns (arithmetic vs motifs vs XOR). Position 5
    in an arithmetic sequence needs different memories than position 5
    in a repeating motif — the router couldn't distinguish them.

    Now the router sees BOTH position q AND token content x,
    so the memory bank is conditioned on what the token IS, not just where.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.dictionary = nn.Parameter(
            torch.randn(cfg.n_dictionary_atoms, cfg.fiber_dim) * 0.02
        )
        self.router = nn.Sequential(
            nn.Linear(cfg.manifold_dim + cfg.fiber_dim, cfg.n_dictionary_atoms),
            nn.SiLU(),
            nn.Linear(cfg.n_dictionary_atoms, cfg.n_dictionary_atoms),
        )

    def forward(self, q: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        """Route based on both position AND content.

        Args:
            q: (batch, manifold_dim) manifold coordinates
            x: (batch, fiber_dim) token states (content signal)
        """
        D_normed = F.normalize(self.dictionary, dim=-1)
        route_input = torch.cat([q, x], dim=-1)
        gate_logits = self.router(route_input)
        _, topk_idx = torch.topk(gate_logits, self.cfg.k_wta, dim=-1)
        M_q = D_normed[topk_idx]
        return M_q

---
## 5. The Causal Score Function

### The v7 Core Innovation

In v1-v6, the Hopfield score function was purely local:  
`score(x_t) = -softmax(beta * x_t^T M) @ M`  
Each position computed its gradient independently.

In v7.1, the score is a sum of **three separate forces** (not blended at input):

```
grad_E      = hopfield_grad(x_local[p], M_p, beta)        # settling force (uncontaminated)
causal_force = context_proj(causal_mix(x_seq)[p] - x[p])  # routing force (cross-position)
inhibition  = gamma * W_inh @ x[p]                        # competitive dynamics
score_p     = grad_E + alpha_step * causal_force + inhibition
```

The causal mixing is a lightweight operation (causal conv via FFT, O(T log T))  
applied at **every Langevin step**. As the states sharpen through annealing,  
the signals flowing through the causal mix become increasingly sparse,  
naturally concentrating routing on shared subspace dimensions.

The score function sees the joint causal state — denoising IS context-aware prediction.

In [36]:
class CausalScoreFunction(nn.Module):
    """Context-aware score function for sequence-level Langevin dynamics.

    === v7.1 FIX (03/13/2026) ===

    Three forces act ADDITIVELY on each token, not blended at input:

    1. Hopfield gradient: computed on the LOCAL state (uncontaminated).
       Pulls toward the nearest memory attractor. This is the settling force.

    2. Causal context force: a learned projection of (x_mixed - x_local).
       Pushes the token in the direction indicated by cross-position context.
       This is the routing force.

    3. Lateral inhibition: competitive dynamics between fiber dimensions.

    KEY MATHEMATICAL FIX: In v7.0, causal context was blended into the
    Hopfield query, corrupting the matching (memory atoms encode single
    tokens, not token+context). Now the forces are separate — the Hopfield
    gradient finds the right attractor, and the causal force steers
    which attractor basin the token falls into.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg

        # Per-fiber decay rates for causal mixing kernel
        self.log_decay = nn.Parameter(
            torch.linspace(-1.0, 2.0, cfg.fiber_dim)
        )

        # Per-step context strength: how much causal force at each Langevin step.
        # Initialized high-to-low: lots of context early (explore), little late (commit).
        init_weights = torch.linspace(1.0, -1.0, cfg.langevin_steps)
        self.step_context_logits = nn.Parameter(init_weights)

        # Learned projection: transforms the causal difference signal
        # (x_mixed - x_local) into a usable gradient direction.
        self.context_proj = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim),
            nn.Tanh(),
        )

        # Lateral inhibition matrix
        self.W_inh = nn.Parameter(
            torch.randn(cfg.fiber_dim, cfg.fiber_dim) * 0.01
        )

    def _causal_mix(self, x_seq: torch.Tensor) -> torch.Tensor:
        """Causal convolution over the sequence dimension via FFT."""
        B, T, D = x_seq.shape

        decay = F.softplus(self.log_decay)
        lags = torch.arange(T, device=x_seq.device).float()
        causal_kernel = torch.exp(-decay.unsqueeze(0) * lags.unsqueeze(-1))
        causal_kernel = causal_kernel / (causal_kernel.sum(dim=0, keepdim=True) + 1e-8)

        x_padded = F.pad(x_seq, (0, 0, 0, T))
        k_padded = F.pad(causal_kernel, (0, 0, 0, T))

        X_seq = torch.fft.fft(x_padded, dim=1)
        K_seq = torch.fft.fft(k_padded, dim=0).unsqueeze(0)

        Y_seq = X_seq * K_seq
        x_mixed = torch.fft.ifft(Y_seq, dim=1).real[:, :T, :]

        return x_mixed

    def hopfield_gradient(self, x, M_q, beta):
        """Hopfield energy gradient on the UNCONTAMINATED local state."""
        similarities = torch.bmm(M_q, x.unsqueeze(-1)).squeeze(-1)
        weights = F.softmax(beta * similarities, dim=-1)
        weighted_memories = torch.bmm(weights.unsqueeze(1), M_q).squeeze(1)
        return -weighted_memories

    def forward(self, x_seq, M_q_all, beta, step_idx):
        """Compute the score as a sum of three separate forces.

        Args:
            x_seq: (B, T, D) current sequence state
            M_q_all: (B*T, k, D) memory banks for all positions
            beta: inverse temperature
            step_idx: which Langevin step (for per-step context weight)

        Returns:
            score: (B, T, D) total denoising direction
        """
        B, T, D = x_seq.shape

        # Force 1: Hopfield gradient on LOCAL state (no context contamination)
        x_flat = x_seq.reshape(B * T, D)
        grad_E = self.hopfield_gradient(x_flat, M_q_all, beta)
        grad_E = grad_E.reshape(B, T, D)

        # Force 2: Causal context force (ADDITIVE, not blended)
        x_mixed = self._causal_mix(x_seq)
        context_diff = x_mixed - x_seq  # what cross-position info adds
        causal_force = self.context_proj(context_diff.reshape(-1, D))
        causal_force = causal_force.reshape(B, T, D)

        # Per-step context strength: high early, low late
        alpha_step = torch.sigmoid(self.step_context_logits[step_idx])
        causal_force = alpha_step * causal_force

        # Force 3: Lateral inhibition
        inhibition = self.cfg.inhibition_gamma * (x_flat @ self.W_inh.T)
        inhibition = inhibition.reshape(B, T, D)

        score = grad_E + causal_force + inhibition
        return score

---
## 6. Sequence-Level Langevin Dynamics

### Denoising the Entire Sequence Simultaneously

The Langevin loop now operates on the full `(B, T, D)` sequence tensor.  
At each step:

1. **Causal score**: three additive forces — Hopfield gradient (settling),  
   causal context force (routing), lateral inhibition (competition).
2. **Langevin update**: gradient descent + annealing noise.
3. **Ramping sparsity**: NO thresholding in early steps (let context flow),  
   linearly increasing to full thresholding in late steps (snap to codes).

Cross-position mixing happens at *every* step of the settling process.  
Early steps (high T): broad exploration with diffuse states.  
Late steps (low T): sharp, selective routing on sparse codes.  
The annealing schedule simultaneously controls settling and routing precision.

In [37]:
class SequenceLangevinDescent(nn.Module):
    """Annealed Langevin dynamics over the full sequence with causal score.

    === v7.1 FIX (03/13/2026) ===

    Two key changes from v7.0:

    1. Passes step_idx to the score function so per-step context weights work.
       Early steps get strong causal force (explore), late steps get weak (commit).

    2. RAMPING SPARSITY: no soft-thresholding in the first half of steps.
       Early steps: let cross-position signals accumulate freely.
       Late steps: hard sparsity enforcement snaps to codes.
       This prevents the proximal operator from killing context signal
       before it has time to steer the token into the right attractor basin.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.score_fn = CausalScoreFunction(cfg)

    def forward(self, x_seq, M_q_all, return_trajectory=False):
        cfg = self.cfg
        x = x_seq.clone()
        trajectory = [x.detach().clone()] if return_trajectory else None
        betas = torch.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)

        # Sparsity ramp: 0 for first half, linearly increasing to full for second half
        sparsity_ramp = torch.clamp(
            torch.linspace(-1.0, 1.0, cfg.langevin_steps), min=0.0
        )

        for step in range(cfg.langevin_steps):
            beta_t = betas[step].item()

            # Score with per-step context weight
            score = self.score_fn(x, M_q_all, beta_t, step)

            # Langevin noise (annealed)
            noise_scale = math.sqrt(2.0 * cfg.langevin_lr / beta_t)
            noise = noise_scale * torch.randn_like(x)

            x = x - cfg.langevin_lr * score + noise

            # Ramping sparsity: no thresholding early, full thresholding late
            lam = cfg.sparsity_lambda * cfg.langevin_lr * sparsity_ramp[step].item()
            if lam > 1e-8:
                x = soft_threshold(x, lam)

            if return_trajectory:
                trajectory.append(x.detach().clone())

        if return_trajectory:
            return x, trajectory
        return x


print("Sequence Langevin Descent v7.1: additive forces + ramping sparsity")

Sequence Langevin Descent v7.1: additive forces + ramping sparsity


---
## 7. The Complete Block and CLM

### v7.2: Three conflict fixes

1. **Gauge transport REMOVED.** It rotated each position into a different  
   coordinate frame, making the causal conv mix incompatible signals.  
   Positional info now comes through the memory bank + positional embeddings.

2. **Gated residual.** Instead of `x_settled + x_seq` (which destroys sparsity),  
   a learned per-dim gate decides how much old state to preserve:  
   `gate * x_settled + (1-gate) * x_seq`.

3. **Content-aware memory bank.** Router sees `[q, x]` (position + content),  
   not just `q`. Different tokens at the same position get different attractors.

In [38]:
class DiffusionRoutingBlock(nn.Module):
    """Sequence-level Langevin settling with content-aware memory bank.

    === v7.2 FIXES ===

    1. REMOVED gauge transport. The gauge rotated each position into a
       different coordinate frame, then the causal conv in the score
       function tried to mix them without accounting for the frame
       differences. They were working against each other. The causal
       mixing now handles cross-position communication (in a single
       shared frame), and positional info comes from the memory bank.

    2. GATED residual. The old residual (x_settled + x_seq) destroyed
       the sparsity that the Langevin loop built. Now the model learns
       a per-dim gate controlling how much of the old state to keep
       vs. how much of the new settled state to trust.

    3. Memory bank receives token content (x) along with position (q)
       so the attractor landscape is content-dependent.
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.memory_bank = DynamicMemoryBank(cfg)
        self.seq_langevin = SequenceLangevinDescent(cfg)
        self.norm = nn.LayerNorm(cfg.fiber_dim)
        self.dropout = nn.Dropout(cfg.dropout)

        # Gated residual: learns per-dim how much old state to preserve
        self.residual_gate = nn.Sequential(
            nn.Linear(cfg.fiber_dim * 2, cfg.fiber_dim),
            nn.Sigmoid(),
        )

    def forward(self, x_seq, q_coords, return_trajectory=False):
        B, T, D = x_seq.shape

        # Memory bank sees both position AND content
        q_flat = q_coords.reshape(B * T, -1)
        x_flat = x_seq.reshape(B * T, D)
        M_q_all = self.memory_bank(q_flat, x_flat)

        # Sequence-level Langevin (unified mix + settle)
        if return_trajectory:
            x_settled, traj = self.seq_langevin(
                x_seq, M_q_all, return_trajectory=True
            )
        else:
            x_settled = self.seq_langevin(x_seq, M_q_all)
            traj = None

        # Gated residual: preserve sparsity of settled state
        gate_input = torch.cat([x_settled, x_seq], dim=-1)
        gate = self.residual_gate(gate_input.reshape(-1, D * 2))
        gate = gate.reshape(B, T, D)
        x_out = self.norm(gate * self.dropout(x_settled) + (1 - gate) * x_seq)

        if return_trajectory:
            return x_out, traj
        return x_out,


class SpectralGaugeCLM(nn.Module):
    """Causal Language Model with Sequence-Level Diffusion Routing.

    === v7.3: Deep supervision — every block gets a direct training signal ===

    The previous versions only had CE loss at the final output. The gradient
    had to backpropagate through 21 chained Langevin steps to reach block 1.
    By that point the signal was almost completely diluted.

    Now: a SHARED decoder reads the output of EVERY block, and each block's
    output contributes a weighted CE loss. This gives every settling stage
    a direct gradient signal — "did your adjustments to the sparse fiber
    relations actually help predict the next token?"

    The decoder is the final spectral layer. The diffusion's job is to
    adjust the relations between sparse states so the decoder can read them
    correctly. Each block should get credit for making representations better.

    Block weights increase with depth (later blocks should be more accurate):
      block 0: weight 0.1 (early, rough)
      block 1: weight 0.3 (mid, refining)
      block 2: weight 1.0 (final, precise)
    """

    def __init__(self, cfg: ArchitectureConfig):
        super().__init__()
        self.cfg = cfg
        self.embedding = SparseTokenEmbedding(cfg)
        self.blocks = nn.ModuleList([
            DiffusionRoutingBlock(cfg) for _ in range(cfg.n_blocks)
        ])
        # SHARED decoder: same projection for all blocks.
        # This forces all blocks to produce representations in the same
        # semantic space — each block refines the SAME representation,
        # not transforming into a new space.
        self.decoder = nn.Sequential(
            nn.Linear(cfg.fiber_dim, cfg.fiber_dim),
            nn.SiLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fiber_dim, cfg.vocab_size),
        )

        # Per-block loss weights (increasing with depth)
        n = cfg.n_blocks
        raw_weights = torch.linspace(0.1, 1.0, n)
        self.register_buffer("block_loss_weights", raw_weights)

    def forward(self, token_ids: torch.Tensor) -> Tuple[torch.Tensor, dict]:
        B, T = token_ids.shape

        x_sparse, q_coords = self.embedding(token_ids)

        x = x_sparse
        intermediate_logits = []

        for block in self.blocks:
            result = block(x, q_coords)
            x = result[0]

            # Decode this block's output for intermediate supervision
            block_logits = self.decoder(x)[:, :-1, :]
            intermediate_logits.append(block_logits)

        info = {
            "mean_sparsity": (x == 0).float().mean().item(),
            "intermediate_logits": intermediate_logits,
            "block_loss_weights": self.block_loss_weights,
        }

        # Final logits = last block's logits
        logits = intermediate_logits[-1]

        return logits, info


model = SpectralGaugeCLM(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")
print(f"\nBlocks: {config.n_blocks}")
print(f"Langevin steps per block: {config.langevin_steps} (each with causal mixing)")
print(f"Block loss weights: {[f'{w:.1f}' for w in model.block_loss_weights.tolist()]}")
print(f"Every block gets a direct training signal via shared decoder")

Total parameters: 1,570,069
Trainable parameters: 1,570,069

Blocks: 3
Langevin steps per block: 7 (each with causal mixing)
Block loss weights: ['0.1', '0.6', '1.0']
Every block gets a direct training signal via shared decoder


---
## 8. Training

In [39]:
def generate_synthetic_data(n_samples, seq_len, vocab_size, n_patterns=8):
    """Same synthetic data as v4-v6 for direct comparison."""
    data = torch.zeros(n_samples, seq_len, dtype=torch.long)

    for i in range(n_samples):
        pattern_type = i % n_patterns

        if pattern_type < 3:
            start = torch.randint(0, vocab_size, (1,)).item()
            step = (pattern_type + 1) * 7
            for t in range(seq_len):
                data[i, t] = (start + t * step) % vocab_size

        elif pattern_type < 5:
            k = pattern_type + 1
            motif = torch.randint(0, vocab_size, (k,))
            for t in range(seq_len):
                data[i, t] = motif[t % k]

        elif pattern_type < 7:
            data[i, 0] = torch.randint(0, vocab_size, (1,))
            data[i, 1] = torch.randint(0, vocab_size, (1,))
            for t in range(2, seq_len):
                data[i, t] = (data[i, t-1].item() ^ data[i, t-2].item()) % vocab_size

        else:
            increments = torch.randint(1, 10, (seq_len,))
            data[i] = torch.cumsum(increments, dim=0) % vocab_size

    return data


seq_len = 16
train_data = generate_synthetic_data(4096, seq_len, config.vocab_size)
val_data = generate_synthetic_data(512, seq_len, config.vocab_size)

print(f"Training data: {train_data.shape}")
print(f"Validation data: {val_data.shape}")
print(f"Sample: {train_data[0].tolist()}")

Training data: torch.Size([4096, 16])
Validation data: torch.Size([512, 16])
Sample: [223, 230, 237, 244, 251, 2, 9, 16, 23, 30, 37, 44, 51, 58, 65, 72]


In [40]:
def train_epoch(model, data, optimizer, cfg):
    """Training with deep supervision: every block gets a direct CE signal.

    The shared decoder reads each block's output. Weighted CE losses give
    every settling stage a direct gradient — not just the final output.
    Early blocks get lighter weight (rough shaping), later blocks get
    heavier weight (precise prediction).
    """
    model.train()
    total_loss = 0.0
    total_ce = 0.0
    n_batches = 0

    perm = torch.randperm(data.size(0))
    data = data[perm]

    for i in range(0, len(data) - cfg.batch_size + 1, cfg.batch_size):
        batch = data[i : i + cfg.batch_size].to(device)
        optimizer.zero_grad()

        logits, info = model(batch)
        targets = batch[:, 1:]

        # Deep supervision: weighted CE from EVERY block
        intermediate_logits = info["intermediate_logits"]
        block_weights = info["block_loss_weights"]

        ce_loss = 0.0
        for block_idx, (block_logits, weight) in enumerate(
            zip(intermediate_logits, block_weights)
        ):
            block_ce = F.cross_entropy(
                block_logits.reshape(-1, cfg.vocab_size), targets.reshape(-1)
            )
            ce_loss = ce_loss + weight * block_ce

        # Normalize by sum of weights so the total scale is comparable
        ce_loss = ce_loss / block_weights.sum()

        # Final block CE for logging (what we actually care about)
        final_ce = F.cross_entropy(
            logits.reshape(-1, cfg.vocab_size), targets.reshape(-1)
        )

        dict_coherence_loss = 0.0
        for block in model.blocks:
            D = F.normalize(block.memory_bank.dictionary, dim=-1)
            gram = D @ D.T
            eye = torch.eye(gram.size(0), device=gram.device)
            dict_coherence_loss = dict_coherence_loss + (gram - eye).pow(2).mean()
        dict_coherence_loss = dict_coherence_loss / len(model.blocks)

        sparsity_loss = logits.abs().mean()

        loss = ce_loss + 0.1 * dict_coherence_loss + 0.01 * sparsity_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_ce += final_ce.item()
        n_batches += 1

    return total_loss / n_batches, total_ce / n_batches


@torch.no_grad()
def evaluate(model, data, cfg):
    model.eval()
    total_ce = 0.0
    total_correct = 0
    total_tokens = 0
    n_batches = 0

    for i in range(0, len(data) - cfg.batch_size + 1, cfg.batch_size):
        batch = data[i : i + cfg.batch_size].to(device)
        logits, info = model(batch)
        targets = batch[:, 1:]

        ce_loss = F.cross_entropy(
            logits.reshape(-1, cfg.vocab_size), targets.reshape(-1)
        )
        total_ce += ce_loss.item()

        preds = logits.argmax(dim=-1)
        total_correct += (preds == targets).sum().item()
        total_tokens += targets.numel()
        n_batches += 1

    avg_ce = total_ce / max(n_batches, 1)
    accuracy = total_correct / max(total_tokens, 1)
    return avg_ce, accuracy


optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=0.02)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.n_epochs)

history = {"train_loss": [], "train_ce": [], "val_ce": [], "val_acc": []}

print("Training Sequence-Level Diffusion Routing CLM (v7)...")
print("=" * 70)

pbar = tqdm(range(config.n_epochs), desc="Training", unit="epoch")
for epoch in pbar:
    train_loss, train_ce = train_epoch(model, train_data, optimizer, config)
    val_ce, val_acc = evaluate(model, val_data, config)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_ce"].append(train_ce)
    history["val_ce"].append(val_ce)
    history["val_acc"].append(val_acc)

    tqdm.write(
        f"Epoch {epoch+1:3d}/{config.n_epochs} | "
        f"Loss: {train_loss:.4f} | "
        f"CE: {train_ce:.4f} | "
        f"Val CE: {val_ce:.4f} | "
        f"Val Acc: {val_acc:.2%}"
    )

print("=" * 70)
print("Training complete.")

Training Sequence-Level Diffusion Routing CLM (v7)...


Training:   0%|          | 0/150 [00:00<?, ?epoch/s]

Epoch   1/150 | Loss: 5.4794 | CE: 5.4786 | Val CE: 5.3703 | Val Acc: 7.10%
Epoch   2/150 | Loss: 5.2030 | CE: 5.1985 | Val CE: 5.0372 | Val Acc: 11.86%
Epoch   3/150 | Loss: 4.7738 | CE: 4.7538 | Val CE: 4.6867 | Val Acc: 15.07%
Epoch   4/150 | Loss: 4.4424 | CE: 4.4155 | Val CE: 4.5736 | Val Acc: 15.03%
Epoch   5/150 | Loss: 4.2785 | CE: 4.2516 | Val CE: 4.5200 | Val Acc: 15.66%
Epoch   6/150 | Loss: 4.1757 | CE: 4.1484 | Val CE: 4.4912 | Val Acc: 16.29%
Epoch   7/150 | Loss: 4.0967 | CE: 4.0676 | Val CE: 4.4766 | Val Acc: 15.85%
Epoch   8/150 | Loss: 4.0310 | CE: 4.0009 | Val CE: 4.4711 | Val Acc: 17.10%
Epoch   9/150 | Loss: 3.9738 | CE: 3.9394 | Val CE: 4.4743 | Val Acc: 17.59%
Epoch  10/150 | Loss: 3.9231 | CE: 3.8863 | Val CE: 4.4541 | Val Acc: 18.12%
Epoch  11/150 | Loss: 3.8716 | CE: 3.8315 | Val CE: 4.4484 | Val Acc: 18.75%
Epoch  12/150 | Loss: 3.8248 | CE: 3.7815 | Val CE: 4.4451 | Val Acc: 19.04%
Epoch  13/150 | Loss: 3.7791 | CE: 3.7327 | Val CE: 4.4510 | Val Acc: 19.69%


---
## 9. Training Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history["train_loss"], 'b-', label='Total Loss')
axes[0, 0].plot(history["train_ce"], 'b--', alpha=0.6, label='CE Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history["val_ce"], 'r-')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Cross-Entropy')
axes[0, 1].set_title('Validation Loss')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history["val_acc"], 'g-')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].set_title('Validation Accuracy')
axes[1, 0].axhline(y=1.0/config.vocab_size, color='gray', linestyle='--',
                    label=f'Random ({1.0/config.vocab_size:.2%})')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

with torch.no_grad():
    D = F.normalize(model.blocks[0].memory_bank.dictionary, dim=-1)
    gram = (D @ D.T).cpu().numpy()

im = axes[1, 1].imshow(gram[:50, :50], cmap='RdBu_r', vmin=-0.3, vmax=0.3)
axes[1, 1].set_title('Dictionary Gram Matrix (first 50 atoms)')
plt.colorbar(im, ax=axes[1, 1], shrink=0.8)

plt.suptitle('v7: Sequence-Level Diffusion Routing — Training Diagnostics',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 10. Autoregressive Generation

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=20):
    model.eval()
    cfg = model.cfg
    device = next(model.parameters()).device
    sequence = prompt.to(device)

    for step in range(max_new_tokens):
        T = sequence.size(1)
        if T >= cfg.max_seq_len:
            break

        logits, _ = model(sequence)
        next_logits = logits[:, -1, :]
        next_token = next_logits.argmax(dim=-1, keepdim=True)
        sequence = torch.cat([sequence, next_token], dim=1)

    return sequence


print("Autoregressive Generation (v7: Diffusion Routing)")
print("=" * 55)

for i in range(5):
    prompt = val_data[i, :4].unsqueeze(0)
    ground_truth = val_data[i].tolist()

    generated = generate(model, prompt, max_new_tokens=12)
    gen_list = generated[0].tolist()

    print(f"\nExample {i+1}:")
    print(f"  Prompt:       {ground_truth[:4]}")
    print(f"  Ground truth: {ground_truth}")
    print(f"  Generated:    {gen_list}")
    matches = sum(a == b for a, b in zip(gen_list[4:], ground_truth[4:]))
    print(f"  Match rate:   {matches}/{min(len(gen_list)-4, len(ground_truth)-4)} correct")

---
## 11. Visualizing the Diffusion-Routing Process

The key diagnostic for v7: how does the settling trajectory look when  
cross-position mixing happens at every Langevin step?

We track:
1. Sparsity evolution across Langevin steps (should sharpen)
2. Cross-position influence at each step (should become more selective)
3. Context weight evolution (how much causal mixing vs. local settling)

In [ ]:
@torch.no_grad()
def visualize_diffusion_routing(model, data, cfg):
    model.eval()
    device = next(model.parameters()).device
    batch = data[:1].to(device)

    x_sparse, q_coords = model.embedding(batch)

    # Run first block with trajectory tracking
    block = model.blocks[0]
    x_out, trajectory = block(x_sparse, q_coords, return_trajectory=True)

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))

    # --- 1. Sparsity evolution across Langevin steps ---
    sparsities = [(t == 0).float().mean().item() for t in trajectory]
    axes[0, 0].plot(sparsities, 'r-o', markersize=4)
    axes[0, 0].set_xlabel('Langevin Step')
    axes[0, 0].set_ylabel('Sparsity Ratio')
    axes[0, 0].set_title('Sparsity Evolution\n(States sharpen → routing sharpens)')
    axes[0, 0].grid(True, alpha=0.3)

    # --- 2. Trajectory heatmap (position 4) ---
    traj_pos4 = torch.stack([t[0, 4, :] for t in trajectory]).cpu().numpy()
    im2 = axes[0, 1].imshow(traj_pos4.T[:64], aspect='auto', cmap='RdBu_r',
                             vmin=-0.3, vmax=0.3)
    axes[0, 1].set_xlabel('Langevin Step')
    axes[0, 1].set_ylabel('Fiber Dimension')
    axes[0, 1].set_title('Position 4: Settling Trajectory\n(Diffused → Sharp Sparse Code)')
    plt.colorbar(im2, ax=axes[0, 1], shrink=0.8)

    # --- 3. Cross-position signal propagation per step ---
    # Measure: how much does changing position 2 affect position 6?
    # Approximate by measuring correlation of changes across steps
    influence_per_step = []
    T = trajectory[0].shape[1]
    for step_idx in range(1, len(trajectory)):
        delta = trajectory[step_idx] - trajectory[step_idx - 1]  # (1, T, D)
        delta_flat = delta[0]  # (T, D)
        norms = delta_flat.norm(dim=-1)  # (T,)
        if norms.sum() > 1e-8:
            delta_normed = delta_flat / (norms.unsqueeze(-1) + 1e-8)
            cross_corr = (delta_normed @ delta_normed.T).abs().mean().item()
        else:
            cross_corr = 0.0
        influence_per_step.append(cross_corr)

    axes[0, 2].plot(range(1, len(trajectory)), influence_per_step, 'b-o', markersize=4)
    axes[0, 2].set_xlabel('Langevin Step')
    axes[0, 2].set_ylabel('Mean Cross-Position Correlation')
    axes[0, 2].set_title('Cross-Position Influence per Step\n(Causal mixing signal strength)')
    axes[0, 2].grid(True, alpha=0.3)

    # --- 4. Initial vs. final state ---
    x_init = trajectory[0][0].cpu().numpy()   # (T, D)
    x_final = trajectory[-1][0].cpu().numpy()  # (T, D)

    im4 = axes[1, 0].imshow(x_init.T[:64], aspect='auto', cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[1, 0].set_xlabel('Position')
    axes[1, 0].set_ylabel('Fiber Dim')
    axes[1, 0].set_title('Initial State (post gauge transport)')
    plt.colorbar(im4, ax=axes[1, 0], shrink=0.8)

    im5 = axes[1, 1].imshow(x_final.T[:64], aspect='auto', cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    axes[1, 1].set_xlabel('Position')
    axes[1, 1].set_ylabel('Fiber Dim')
    axes[1, 1].set_title('Final State (after sequence-level settling)')
    plt.colorbar(im5, ax=axes[1, 1], shrink=0.8)

    # --- 6. Temperature schedule ---
    betas = np.linspace(cfg.beta_init, cfg.beta_final, cfg.langevin_steps)
    temps = 1.0 / betas
    ax_twin = axes[1, 2]
    ax_twin.plot(temps, 'purple', linewidth=2, label='Temperature (1/beta)')
    ax_twin.set_xlabel('Langevin Step')
    ax_twin.set_ylabel('Temperature', color='purple')
    ax_twin.set_title('Annealing Schedule\n(Controls BOTH settling AND routing)')
    ax_twin.grid(True, alpha=0.3)

    # Overlay per-step context weights
    step_alphas = torch.sigmoid(block.seq_langevin.score_fn.step_context_logits).cpu().numpy()
    ax2 = ax_twin.twinx()
    ax2.plot(step_alphas, 'g-o', markersize=4, label='Context strength')
    ax2.set_ylabel('Context Strength (alpha)', color='green')
    ax2.set_ylim(0, 1)
    ax_twin.legend(loc='upper left')
    ax2.legend(loc='upper right')

    plt.suptitle('v7: Anatomy of Sequence-Level Diffusion Routing',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


visualize_diffusion_routing(model, val_data, config)

---
## 12. Summary

### v7 Architecture: Sequence-Level Diffusion Routing

```
Input Token IDs: [t_1, t_2, ..., t_T]
        |
        v
[Sparse Token Embedding] → sparse fiber sections + manifold coordinates
        |
        v
For each DiffusionRoutingBlock (x n_blocks):
  |
  [GAUGE TRANSPORT] Per-fiber FFT: advection-diffusion
  |   Wilson line holonomy accumulated
  |
  [SEQUENCE-LEVEL LANGEVIN] (the v7 core)
  |   For each Langevin step (high T → low T):
  |     1. Causal conv (FFT over seq dim) — mix across positions
  |     2. Blend: alpha * mixed + (1-alpha) * local
  |     3. Hopfield gradient on context-enriched state
  |     4. Lateral inhibition
  |     5. Langevin update: x = x - eta*score + noise
  |     6. Soft-threshold → enforce sparsity
  |   ↓
  |   Each step: mixing gets sharper as states get sparser
  |   Denoising IS routing. Same process.
  |
  [Residual + LayerNorm]
        |
        v
[Decoder] → logits → P(t_{t+1} | t_1, ..., t_t)
```

### What v7 Unifies

v1-v6 had separate stages for cross-position mixing and per-token settling.  
v7 fuses them: the score function at each Langevin step includes causal mixing.  

| Previous (v4-v6) | v7 |
|---|---|
| Mix → Settle (two stages) | **Mix INSIDE Settle (one process)** |
| Cross-position info applied once | **Refined at every Langevin step** |
| Routing and settling are independent | **Routing sharpens as states sharpen** |
| Separate parameters for mixing vs. settling | **Score function handles both** |

### The Insight

Diffusion models are built to denoise by learning the intricate  
sub-interactions in the data. We define a score function across the  
sequence that does exactly this: it learns which cross-position  
interactions matter for predicting the clean sparse state at each position.

The annealing schedule now controls two things simultaneously:  
- **Sparsity** (weak activations → zero)  
- **Routing selectivity** (sparse signals → selective mixing)  

They are the same phenomenon.

---
*v7 notebook by David Ledbetter, 03/13/2026.*  
*Sequence-level diffusion routing: denoising is cross-position mixing.*